In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [3]:
load_dotenv()
llm = ChatOpenAI()

In [4]:
class JokeStat(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeStat):

    prompt = f'generate a joke on the topic {state['topic']}'
    res = llm.invoke(prompt).content

    return {'joke': res}

In [13]:
def generate_explanation(state: JokeStat):

    prompt = f'generate a short explanation for the given joke \n Joke -> {state['joke']}'
    res = llm.invoke(prompt).content

    return {'explanation': res}

In [14]:
graph = StateGraph(JokeStat)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

joke_graph = graph.compile(checkpointer=checkpointer)

In [15]:
config1 = {"configurable": {"thread_id": "1"}}
joke_graph.invoke({'topic': 'burger'}, config=config1)

{'topic': 'burger',
 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!',
 'explanation': 'This joke plays on the double meaning of "buns" - referring to both the burger buns and someone\'s glute muscles. The burger went to the gym to work on its "buns," or its bun buns, to improve its overall appearance.'}

In [16]:
joke_graph.get_state(config1)

StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!', 'explanation': 'This joke plays on the double meaning of "buns" - referring to both the burger buns and someone\'s glute muscles. The burger went to the gym to work on its "buns," or its bun buns, to improve its overall appearance.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6634-3e64-6a82-8002-05788db2aa4e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-12T17:35:06.842368+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6634-3610-6dc2-8001-4f5ed48e0efb'}}, tasks=(), interrupts=())

In [17]:
list(joke_graph.get_state_history(config1))

[StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!', 'explanation': 'This joke plays on the double meaning of "buns" - referring to both the burger buns and someone\'s glute muscles. The burger went to the gym to work on its "buns," or its bun buns, to improve its overall appearance.'}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6634-3e64-6a82-8002-05788db2aa4e'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-10-12T17:35:06.842368+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6634-3610-6dc2-8001-4f5ed48e0efb'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'burger', 'joke': 'Why did the burger go to the gym?\n\nTo get better buns!'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1c6634-3610-6dc2-8001-4f5ed48e0efb'}},